[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch05-nn-computation/02-so-hoc-va-ham-kich-hoat.ipynb)

# Số học không nhánh vẫn có rủi ro: tràn số, hàm kích hoạt và entropy chéo

Chương 5 mở đầu bằng một tên lửa hỏng vì một phép đổi số. Rồi chương nói rằng mạng nơ-ron gặp
đúng loại hỏng đó: phần tính toán của mô hình không có một nhánh logic nào, nhưng một giá trị
vượt miền biểu diễn vẫn thành vô cực hay NaN, và lan đi mà không dừng chương trình.

Sổ tay này đo những chỗ hỏng đó, rồi xem các hàm kích hoạt và hàm mất mát của chương được dựng
ra sao để tránh chúng.

**Bạn sẽ làm:**
1. lặp lại phép đổi 64 bit sang 16 bit của Ariane 5, một lần báo lỗi, một lần im lặng;
2. đo miền giá trị của FP32 và FP16, rồi xem một giá trị NaN lan khắp mô hình sau một bước cập nhật;
3. tính sigmoid, tanh, ReLU và đạo hàm của chúng bằng autograd, rồi xem gradient co lại qua mười tầng;
4. đặt thuế transistor của chương cạnh thời gian chạy đo trên CPU;
5. kiểm vì sao cần tính phi tuyến, trên bài toán XOR;
6. dựng softmax và entropy chéo ổn định về số, rồi kiểm gradient $p - y$.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import struct

import torch.nn.functional as F

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # bốn màu cố định cho các chuỗi số liệu

## 1. Một phép đổi số, hai cách hỏng

Năm 1996, phần mềm dẫn đường quán tính của Ariane 5, dùng lại từ Ariane 4, đổi một giá trị dấu
phẩy động 64 bit sang số nguyên có dấu 16 bit. Quỹ đạo của tên lửa mới khiến giá trị đó vượt
miền của số nguyên 16 bit. Phép đổi sinh ra lỗi toán hạng, và không đoạn mã nào xử lý lỗi đó.

Một số nguyên có dấu 16 bit chỉ có $2^{16}$ giá trị, từ $-2^{15}$ đến $2^{15}-1$.

In [ ]:
BIT_NGUON = sach(64, nguon="64 bit → 16 bit")
BIT_DICH = sach(16, nguon="64 bit → 16 bit")
GIAY = sach(36, nguon="36 giây")
THIET_HAI = sach(370e6, nguon="370 triệu đô la")
print(f"Ariane 5, chuyến bay 501: đổi {BIT_NGUON} bit → {BIT_DICH} bit, máy tính bay ngừng sau {GIAY} giây, "
      f"mất khối vệ tinh {THIET_HAI / 1e6:.0f} triệu đô la")

nho_nhat, lon_nhat = -2 ** (BIT_DICH - 1), 2 ** (BIT_DICH - 1) - 1
assert (nho_nhat, lon_nhat) == (np.iinfo(np.int16).min, np.iinfo(np.int16).max)
print(f"số nguyên có dấu {BIT_DICH} bit: từ {nho_nhat:,} đến {lon_nhat:,}")

### Dự đoán

Đưa một giá trị ví dụ, 40,000 (con số của sổ tay này, không phải của chuyến bay), vào số nguyên
16 bit. Thư viện sẽ báo lỗi, trả về giá trị lớn nhất 32,767, hay trả về một số khác hẳn? Hãy
đoán cho cả hai thư viện bên dưới trước khi chạy.

In [ ]:
gia_tri = 40_000.0  # giá trị ví dụ, nằm ngoài miền của int16

# Cách 1: thư viện kiểm tra miền rồi báo lỗi, giống phép đổi trên Ariane 5.
try:
    struct.pack("<h", int(gia_tri))
    print("struct: đổi được")
except struct.error as loi:
    print("struct báo lỗi:", loi)

# Cách 2: NumPy và PyTorch giữ 16 bit thấp của số nguyên 64 bit, và không báo gì cả.
im_lang = int(np.array([int(gia_tri)], dtype=np.int64).astype(np.int16)[0])
print(f"NumPy trả về {im_lang:,}, không một lời cảnh báo")
assert im_lang == int(gia_tri) - 2 ** BIT_DICH
assert torch.tensor([int(gia_tri)]).to(torch.int16).item() == im_lang

Hai cách hỏng khác hẳn nhau. Ở cách thứ nhất, lỗi được báo ngay; trên Ariane 5 lỗi đó không có
ai bắt, và máy tính bay ngừng hoạt động. Ở cách thứ hai, không có gì dừng lại cả: chương trình
chạy tiếp với một số âm, đúng cú pháp nhưng sai hoàn toàn với chế độ vật lý mà nó đang chạy.

Sổ tay đi qua số nguyên 64 bit vì một lý do: đổi thẳng một số thực nằm ngoài miền sang int16
là hành vi không xác định trong C, nên kết quả tuỳ bộ xử lý. Đi qua int64 thì máy nào cũng cho
cùng đáp số.

Học máy gặp cách hỏng thứ hai thường xuyên hơn: phần cứng âm thầm trả về vô cực hay giá trị
không hợp lệ, rồi các phép tính sau cứ thế dùng nó.

## 2. Miền giá trị của FP32 và FP16

Số dấu phẩy động cũng có miền. Giá trị nhỏ hơn số chuẩn dương nhỏ nhất rơi vào vùng số dưới
chuẩn (subnormal), rồi bị làm tròn về 0: đó là tràn dưới (underflow). Giá trị lớn hơn mức lớn
nhất biểu diễn được thì thành vô cực, Inf: đó là tràn trên (overflow).

In [ ]:
for kieu in (np.float32, np.float16):
    fi = np.finfo(kieu)
    print(f"{fi.dtype}: chuẩn nhỏ nhất {float(fi.tiny):.3e} · dưới chuẩn nhỏ nhất "
          f"{float(fi.smallest_subnormal):.3e} · lớn nhất {float(fi.max):,.6g}")

theo_sach("số chuẩn dương nhỏ nhất của FP32", float(np.finfo(np.float32).tiny),
          sach=1.18e-38, nguon="1,18×10⁻³⁸ (FP32)")
theo_sach("số chuẩn dương nhỏ nhất của FP16", float(np.finfo(np.float16).tiny),
          sach=6.10e-5, nguon="6,10×10⁻⁵ (FP16)")
print(f"số lớn nhất của FP16 nhỏ hơn của FP32 {float(np.finfo(np.float32).max) / float(np.finfo(np.float16).max):.1e} lần")

### Tràn dưới: một gradient nhỏ dần

Lan truyền ngược nhân gradient với một thừa số ở mỗi tầng. Giả sử (một kịch bản minh hoạ) mỗi
tầng nhân gradient với 0.1, bắt đầu từ 0.01. FP32 giữ được giá trị đó qua rất nhiều tầng. FP16
thì sao?

In [ ]:
so_tang = 12
g16, g32 = [], []
a16, a32 = torch.tensor(1e-2, dtype=torch.float16), torch.tensor(1e-2, dtype=torch.float32)
for _ in range(so_tang):
    a16, a32 = a16 * 0.1, a32 * 0.1
    g16.append(a16.item())
    g32.append(a32.item())

tang_ve_0 = next(k for k, g in enumerate(g16, 1) if g == 0.0)
print(f"FP16 rơi xuống 0 sau {tang_ve_0} tầng; FP32 sau {so_tang} tầng vẫn còn {g32[-1]:.1e}")

fig, ax = plt.subplots(figsize=(7, 3.8))
tang = np.arange(1, so_tang + 1)
ax.semilogy(tang, g32, "o-", color=MAU[0], lw=2, label="FP32")
duong = np.array(g16)
ax.semilogy(tang[duong > 0], duong[duong > 0], "s-", color=MAU[1], lw=2, label="FP16 (khác 0)")
ax.plot(tang[duong == 0], np.full((duong == 0).sum(), 3e-15), "x", color=MAU[1], ms=9,
        label="FP16 đã về 0 (vẽ ở đáy)")
chuan16 = float(np.finfo(np.float16).tiny)
duoi_chuan16 = float(np.finfo(np.float16).smallest_subnormal)
ax.axhspan(duoi_chuan16, chuan16, color="0.85", alpha=0.6, lw=0)
ax.text(so_tang + 0.4, (chuan16 * duoi_chuan16) ** 0.5, "vùng số dưới chuẩn\ncủa FP16",
        va="center", fontsize=8, color="0.3")
ax.text(so_tang + 0.4, duoi_chuan16 / 30, "nhỏ hơn nữa:\nFP16 về 0", va="center", fontsize=8, color="0.3")
ax.set_xlim(0.5, so_tang + 3.6)
ax.set_xticks(tang)
ax.set_xlabel("số tầng gradient đã đi qua")
ax.set_ylabel("độ lớn gradient")
ax.set_title("Mỗi tầng nhân gradient với 0.1 (kịch bản minh hoạ)")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

FP16 đi qua vùng số dưới chuẩn rồi về hẳn 0 chỉ sau vài tầng. Từ đó, mọi trọng số phía trước
nhận gradient bằng 0 và đứng yên, và không có thông báo nào.

### Tràn trên: một NaN lan khắp mô hình

Ở đầu kia, $e^x$ vượt mức lớn nhất của FP16 khi $x$ chỉ hơn 11 một chút, vì
$\ln 65504 \approx 11.09$. Vô cực trừ vô cực thì ra NaN.

In [ ]:
for x in (10.0, 11.0, 11.1, 12.0):
    print(f"e^{x} ở FP16 = {torch.exp(torch.tensor(x, dtype=torch.float16)).item()}")
print("ngưỡng ln(lớn nhất): FP16", round(math.log(float(np.finfo(np.float16).max)), 2),
      "· FP32", round(math.log(float(np.finfo(np.float32).max)), 2))
vo_cuc = torch.exp(torch.tensor(12.0, dtype=torch.float16))
print("Inf − Inf =", (vo_cuc - vo_cuc).item())

### Dự đoán

Một batch 8 mẫu đi qua một tầng kết nối đầy đủ, và **một** giá trị của **một** mẫu là NaN. Sau
một bước cập nhật, bao nhiêu trọng số của tầng bị hỏng? Rồi một batch hoàn toàn sạch đi qua
tầng đó: bao nhiêu đầu ra là NaN?

In [ ]:
dat_hat_giong()
tang_fc = torch.nn.Linear(4, 3)
x = torch.randn(8, 4)
x[5, 2] = float("nan")  # một giá trị hỏng, trong một mẫu
y = torch.randint(0, 3, (8,))

mat_mat = F.cross_entropy(tang_fc(x), y)  # không có lỗi nào được báo
mat_mat.backward()
print("mất mát của batch:", mat_mat.item())
print(f"gradient NaN: {torch.isnan(tang_fc.weight.grad).sum().item()} / {tang_fc.weight.numel()} trọng số")

with torch.no_grad():
    for p in tang_fc.parameters():
        p -= 0.1 * p.grad  # một bước hạ gradient
    dau_ra = tang_fc(torch.randn(8, 4))  # một batch sạch
print(f"đầu ra NaN trên batch sạch: {torch.isnan(dau_ra).sum().item()} / {dau_ra.numel()}")
assert torch.isnan(tang_fc.weight).all() and torch.isnan(dau_ra).all()

Gradient trọng số là tổng trên mọi mẫu của batch, nên một mẫu hỏng làm hỏng cả ma trận gradient.
Sau một bước cập nhật, mọi trọng số là NaN, và mô hình trả lời NaN cho mọi đầu vào về sau, kể cả
đầu vào sạch. Suốt quá trình đó không có ngoại lệ nào: đó là ý của chương khi nói chế độ số quan
trọng với việc gỡ lỗi không kém gì luồng điều khiển.

## 3. Ba hàm kích hoạt theo từng phần tử, và đạo hàm của chúng

Chương viết ba hàm kích hoạt (activation function) tác động theo từng phần tử:

* sigmoid, $\sigma(x) = \dfrac{1}{1+e^{-x}}$, nén đầu vào vào khoảng $(0, 1)$;
* tanh, $\tanh(x) = \dfrac{e^x - e^{-x}}{e^x + e^{-x}}$, đối xứng quanh 0 trong khoảng $(-1, 1)$;
* ReLU (đơn vị tuyến tính chỉnh lưu), $\max(0, x)$.

Bên dưới, mỗi hàm được viết đúng theo công thức của chương, đối chiếu với hàm có sẵn của PyTorch,
rồi lấy đạo hàm bằng autograd thay vì bằng tay.

In [ ]:
ham = {
    "sigmoid": lambda x: 1 / (1 + torch.exp(-x)),
    "tanh": lambda x: (torch.exp(x) - torch.exp(-x)) / (torch.exp(x) + torch.exp(-x)),
    "ReLU": lambda x: torch.where(x > 0, x, torch.zeros_like(x)),
}
co_san = {"sigmoid": torch.sigmoid, "tanh": torch.tanh, "ReLU": torch.relu}

truc_x = torch.linspace(-6, 6, 241, dtype=torch.float64)
gia_tri_ham, dao_ham = {}, {}
for ten, f in ham.items():
    x = truc_x.clone().requires_grad_(True)
    y = f(x)
    y.sum().backward()  # autograd: đạo hàm của từng phần tử theo đầu vào của nó
    gia_tri_ham[ten], dao_ham[ten] = y.detach(), x.grad
    assert torch.allclose(y.detach(), co_san[ten](truc_x))

# Đạo hàm theo công thức đóng, để kiểm autograd.
s = torch.sigmoid(truc_x)
assert torch.allclose(dao_ham["sigmoid"], s * (1 - s))
assert torch.allclose(dao_ham["tanh"], 1 - torch.tanh(truc_x) ** 2)
assert torch.equal(dao_ham["ReLU"], (truc_x > 0).double())
print("autograd khớp công thức đóng ở cả ba hàm")

### Dự đoán

Ở $x = 0$, đạo hàm của hàm nào lớn nhất? Còn ở $x = 5$, khi đầu vào đã lớn?

In [ ]:
for x0 in (0.0, 2.0, 5.0, 10.0):
    t = torch.tensor(x0, dtype=torch.float64)
    print(f"x = {x0:>4}: σ' = {(torch.sigmoid(t) * (1 - torch.sigmoid(t))).item():.2e} · "
          f"tanh' = {(1 - torch.tanh(t) ** 2).item():.2e} · ReLU' = {float(x0 > 0):.0f}")

dinh_sigmoid = dao_ham["sigmoid"].max().item()
assert abs(dinh_sigmoid - 0.25) < 1e-12  # σ'(0) = σ(0)(1 − σ(0)) = 1/2 · 1/2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
for (ten, gt), mau in zip(gia_tri_ham.items(), MAU):
    ax1.plot(truc_x, gt, color=mau, lw=2, label=ten)
    dh = dao_ham[ten].clone()
    if ten == "ReLU":
        dh[truc_x.abs() < 0.03] = float("nan")  # bước nhảy ở 0: không nối hai nhánh bằng một đoạn dốc
    ax2.plot(truc_x, dh, color=mau, lw=2, label=ten)
ax1.set_ylim(-1.5, 3)
ax1.set_title("Hàm kích hoạt")
ax2.set_title("Đạo hàm (autograd)")
ax2.annotate(f"đỉnh của σ': {dinh_sigmoid:.2f}", (0, dinh_sigmoid), xytext=(1.6, 0.42),
             arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8), fontsize=8)
ax2.annotate("bão hoà: đạo hàm gần 0", (-5, 0.01), xytext=(-6, 0.62), fontsize=8,
             arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
for ax in (ax1, ax2):
    ax.set_xlabel("x")
    ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Đạo hàm của sigmoid không bao giờ vượt quá giá trị của nó tại 0, tức một phần tư, và gần như
bằng 0 khi đầu vào có trị tuyệt đối lớn. Tanh lên tới 1 ở 0, nhưng cũng bão hoà ở hai đầu.
ReLU có đạo hàm bằng đúng 1 trên toàn bộ nhánh dương và bằng 0 trên nhánh âm. Tại đúng điểm 0,
ReLU không có đạo hàm; PyTorch gán cho nó một giá trị quy ước, ở đây là 0, như chương nói về
đạo hàm dưới (subgradient) được gán tại 0.

### Qua nhiều tầng

Lan truyền ngược nhân các đạo hàm đó với nhau, mỗi tầng một lần. Phần này chỉ lướt qua hệ quả,
bằng hai phép tính: một cận trên, và một phép đo trên một mạng mười tầng khởi tạo ngẫu nhiên.

In [ ]:
so_tang = 10
print(f"cận trên của tích đạo hàm sigmoid qua {so_tang} tầng: {dinh_sigmoid ** so_tang:.1e}")

chuan_gradient = {}
for ten, lop_kich_hoat in (("sigmoid", torch.nn.Sigmoid), ("ReLU", torch.nn.ReLU)):
    dat_hat_giong()
    cac_lop = []
    for _ in range(so_tang):
        cac_lop += [torch.nn.Linear(64, 64), lop_kich_hoat()]
    mang = torch.nn.Sequential(*cac_lop, torch.nn.Linear(64, 10))
    F.cross_entropy(mang(torch.randn(32, 64)), torch.randint(0, 10, (32,))).backward()
    chuan_gradient[ten] = [mang[2 * i].weight.grad.norm().item() for i in range(so_tang)]

fig, ax = plt.subplots(figsize=(7, 3.8))
for (ten, cg), mau in zip(chuan_gradient.items(), MAU):
    ax.semilogy(range(1, so_tang + 1), cg, "o-", color=mau, lw=2, label=ten)
    ax.annotate(ten, (1, cg[0]), xytext=(-8, 0), textcoords="offset points", ha="right",
                va="center", fontsize=8, color="0.2")
ax.set_xlim(0, so_tang + 0.5)
ax.set_xticks(range(1, so_tang + 1))
ax.set_xlabel("tầng (1 là tầng gần đầu vào nhất)")
ax.set_ylabel("chuẩn gradient của trọng số")
ax.set_title("Một lượt truyền ngược qua mười tầng, khởi tạo ngẫu nhiên")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()
print(f"tỉ lệ tầng cuối / tầng đầu: sigmoid {chuan_gradient['sigmoid'][-1] / chuan_gradient['sigmoid'][0]:.1e}"
      f" · ReLU {chuan_gradient['ReLU'][-1] / chuan_gradient['ReLU'][0]:.1e}")

Với sigmoid, gradient ở tầng đầu nhỏ hơn ở tầng cuối nhiều bậc độ lớn: đó là tiêu biến gradient
(vanishing gradient), và cập nhật ở các tầng đầu trở nên không đáng kể. Với ReLU, khoảng chênh
nhỏ hơn hẳn. Chương nhắc rằng gradient của cả mạng vẫn có thể co hay giãn qua trọng số, nên đây
là một xu hướng, không phải lời bảo đảm.

### ReLU sinh ra số 0, nhưng số 0 không tự chạy nhanh hơn

ReLU đưa mọi giá trị âm về 0, nên giá trị kích hoạt của nó thưa. Một kernel nhân ma trận dày
đặc vẫn xử lý từng vị trí bằng 0 như mọi vị trí khác. Thử đo.

In [ ]:
dat_hat_giong()
kich_hoat = torch.relu(torch.randn(256, 1024))
print(f"tỉ lệ số 0 sau ReLU: {(kich_hoat == 0).float().mean().item():.1%}")


def thoi_gian(f, lap=30):
    """Trung vị thời gian của f(), sau một lần chạy khởi động."""
    f()
    do = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        do.append(time.perf_counter() - t0)
    return float(np.median(do))


W = torch.randn(1024, 1024)
dau_vao_day = torch.randn(256, 1024)
day_dac = thoi_gian(lambda: dau_vao_day @ W)
thua = thoi_gian(lambda: kich_hoat @ W)
do_tren_may("nhân ma trận, đầu vào dày đặc", day_dac * 1e3, "ms")
do_tren_may("nhân ma trận, đầu vào nửa số 0", thua * 1e3, "ms")
do_tren_may("tỉ lệ thời gian (nửa số 0 / dày đặc)", thua / day_dac)

Hai thời gian gần như bằng nhau: số phép tính danh nghĩa giảm một nửa, nhưng kernel dày đặc
không biết điều đó. Chỉ khi hiện thực khai thác được độ thưa, chẳng hạn độ thưa có cấu trúc (structured
sparsity) mà phần cứng hỗ trợ, số 0 mới thành thời gian tiết kiệm.

## 4. Thuế transistor, và thời gian đo trên CPU

Chương còn tính một cái giá thứ hai của hàm kích hoạt: diện tích silicon. Một datapath ReLU chỉ
cần logic so sánh và chọn, còn sigmoid hay tanh cần xấp xỉ một hàm mũ. Theo các giả định minh
hoạ của chương:

In [ ]:
RELU_TRANSISTOR = sach(50, nguon="50 vs 2.500 transistor-tương-đương")
MU_TRANSISTOR = sach(2500, nguon="50 vs 2.500 transistor-tương-đương")
theo_sach("thuế transistor, sigmoid so với ReLU", MU_TRANSISTOR / RELU_TRANSISTOR, sach=50,
          nguon="transistor-tương-đương · 50×")
print(f"ReLU: {RELU_TRANSISTOR} transistor tương đương · datapath hàm mũ: {MU_TRANSISTOR:,}")

### Dự đoán

Trên CPU của bạn, chạy sigmoid trên một triệu giá trị chậm hơn ReLU bao nhiêu lần: khoảng 50
lần như thuế transistor, ít hơn, hay nhiều hơn? Để có mốc, ô dưới đo cả thời gian chỉ chép mảng,
tức không tính gì cả.

In [ ]:
N = 2 ** 20
x = torch.randn(N)
do_kich_hoat = {
    "chép mảng": thoi_gian(lambda: x.clone()),
    "ReLU": thoi_gian(lambda: torch.relu(x)),
    "sigmoid": thoi_gian(lambda: torch.sigmoid(x)),
    "tanh": thoi_gian(lambda: torch.tanh(x)),
}
for ten, t in do_kich_hoat.items():
    do_tren_may(f"{ten}, mỗi phần tử", t / N * 1e9, "ns")
do_tren_may("sigmoid / ReLU", do_kich_hoat["sigmoid"] / do_kich_hoat["ReLU"], "lần")

# ReLU chạy riêng: một phép so sánh trên mỗi phần tử, đọc 4 byte và ghi 4 byte ở FP32.
BYTE_FP32 = 4
mat_do_relu = 1 / (2 * BYTE_FP32)
theo_sach("mật độ tính toán của ReLU chạy riêng", mat_do_relu, sach=0.125, nguon="0,125 FLOP/byte (FP32)")
do_tren_may("băng thông hiệu dụng của ReLU", 2 * BYTE_FP32 * N / do_kich_hoat["ReLU"] / 1e9, "GB/s")

fig, ax = plt.subplots(figsize=(7, 3.2))
ten_cot = list(do_kich_hoat)
ns = [do_kich_hoat[k] / N * 1e9 for k in ten_cot]
ax.barh(ten_cot, ns, color=MAU[0], height=0.55)
for i, v in enumerate(ns):
    ax.text(v, i, f" {v:.2f} ns", va="center", fontsize=8)
ax.set_xlim(0, max(ns) * 1.25)
ax.invert_yaxis()
ax.set_xlabel("ns mỗi phần tử (đo trên máy này)")
ax.set_title("Thời gian mỗi phần tử, trên một triệu giá trị FP32")
plt.tight_layout()
plt.show()

Tỉ lệ đo được trên CPU không phải 50, và không cần phải là 50: thuế transistor là diện tích của
một mạch phần cứng theo giả định minh hoạ, còn đây là thời gian của một chương trình trên một
bộ xử lý đa dụng. Chương nói rõ tỉ lệ chính xác không phải một hằng số phần cứng.

Điều đáng để ý là ReLU: nếu thời gian của nó gần thời gian chép mảng, thì ReLU chạy riêng đang
nghẽn ở bộ nhớ, đúng như mật độ 0.125 FLOP mỗi byte dự báo. Vì thế người ta hợp nhất hàm kích
hoạt với kernel nhân ma trận đứng trước nó, để khỏi tốn thêm một chuyến đi về bộ nhớ.

## 5. Vì sao cần tính phi tuyến

Thiếu hàm kích hoạt, hai tầng tuyến tính sụp lại thành một: $(\mathbf{x}W_1 + b_1)W_2 + b_2 =
\mathbf{x}(W_1W_2) + (b_1W_2 + b_2)$. Kiểm bằng số:

In [ ]:
dat_hat_giong()
t1, t2 = torch.nn.Linear(784, 128).double(), torch.nn.Linear(128, 10).double()
x = torch.rand(32, 784, dtype=torch.float64)
W_gop = t1.weight.T @ t2.weight.T  # 784 × 10
b_gop = t1.bias @ t2.weight.T + t2.bias
assert torch.allclose(t2(t1(x)), x @ W_gop + b_gop)
print(f"hai tầng tuyến tính = một ma trận {tuple(W_gop.shape)}: "
      f"{t1.weight.numel() + t2.weight.numel():,} trọng số mà chỉ biểu diễn được {W_gop.numel():,}")

### XOR

Bài toán XOR của chương: hai đầu vào bằng 0 hoặc 1, đầu ra bằng 1 khi hai đầu vào khác nhau.

**Dự đoán:** một bộ phân loại tuyến tính, tức một đường thẳng chia mặt phẳng, phân loại đúng
nhiều nhất bao nhiêu trong bốn điểm?

In [ ]:
X_xor = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = torch.tensor([0.0, 1.0, 1.0, 0.0])

# Vét cạn mọi đường thẳng w1·x1 + w2·x2 + b = 0 trên một lưới dày.
luoi = torch.linspace(-2, 2, 41)
w1, w2, b = torch.meshgrid(luoi, luoi, luoi, indexing="ij")
du_doan = (w1[..., None] * X_xor[:, 0] + w2[..., None] * X_xor[:, 1] + b[..., None]) > 0
tot_nhat = (du_doan.float() == y_xor).float().mean(-1).max().item()
print(f"đường thẳng tốt nhất phân loại đúng {tot_nhat:.0%} số điểm")
assert tot_nhat == 0.75


# Mạng của chương: hai đầu vào → hai nơ-ron ẩn dùng ReLU → một đầu ra dùng sigmoid.
# Trọng số đặt bằng tay: h1 = ReLU(x1 + x2), h2 = ReLU(x1 + x2 − 1), z = 2·h1 − 4·h2 − 1.
def xor_tay(X):
    h1 = torch.relu(X[:, 0] + X[:, 1])
    h2 = torch.relu(X[:, 0] + X[:, 1] - 1)
    return torch.sigmoid(2 * h1 - 4 * h2 - 1)


assert torch.equal((xor_tay(X_xor) > 0.5).float(), y_xor)
print("mạng 2-2-1 đặt tay phân loại đúng cả bốn điểm:", xor_tay(X_xor).numpy().round(3))

Mạng có tầng ẩn biểu diễn được XOR. Nhưng chương nhắc rằng biểu diễn được không có nghĩa là huấn
luyện sẽ tìm ra. Huấn luyện cùng mạng 2-2-1 từ mười hạt giống khác nhau, rồi đếm:

In [ ]:
thanh_cong = []
for seed in range(10):
    torch.manual_seed(seed)
    mang = torch.nn.Sequential(torch.nn.Linear(2, 2), torch.nn.ReLU(), torch.nn.Linear(2, 1))
    toi_uu = torch.optim.Adam(mang.parameters(), lr=0.05)
    for _ in range(1500):
        mat_mat = F.binary_cross_entropy_with_logits(mang(X_xor).squeeze(1), y_xor)
        toi_uu.zero_grad()
        mat_mat.backward()
        toi_uu.step()
    with torch.no_grad():
        dung = bool(((mang(X_xor).squeeze(1) > 0).float() == y_xor).all())
        chet = int((torch.relu(mang[0](X_xor)).sum(0) == 0).sum())
    thanh_cong.append(dung)
    print(f"hạt giống {seed}: {'giải được' if dung else 'không giải được'}, {chet} nơ-ron ẩn chết")
print(f"giải được XOR: {sum(thanh_cong)} / {len(thanh_cong)} hạt giống")

Với chỉ hai nơ-ron ẩn, một nơ-ron ReLU chết, tức cho ra 0 với cả bốn đầu vào, là đủ để lần chạy
đó không bao giờ giải được bài toán. Tầng ẩn đổi lớp hàm mà mạng biểu diễn được; còn tìm ra hàm
đó hay không thì tuỳ khởi tạo và tối ưu.

## 6. Softmax và entropy chéo, viết cho ổn định về số

Softmax biến $K$ logit thành $K$ xác suất có tổng bằng 1:
$\text{softmax}(z_i) = \dfrac{e^{z_i}}{\sum_j e^{z_j}}$. Trừ cùng một số khỏi mọi logit không đổi
kết quả, vì thừa số chung triệt tiêu giữa tử và mẫu. Chương trừ đi logit lớn nhất, để luỹ thừa
lớn nhất luôn là $e^0 = 1$.

In [ ]:
def softmax_ngay_tho(z):
    e = torch.exp(z)
    return e / e.sum(-1, keepdim=True)


def softmax_on_dinh(z):
    e = torch.exp(z - z.max(-1, keepdim=True).values)
    return e / e.sum(-1, keepdim=True)


z = torch.tensor([2.0, 1.0, 0.1])
assert torch.allclose(softmax_ngay_tho(z), softmax_on_dinh(z))
assert torch.allclose(softmax_on_dinh(z), softmax_on_dinh(z + 100))
print("softmax", softmax_on_dinh(z).numpy().round(4), "· tổng", round(softmax_on_dinh(z).sum().item(), 6))

### Dự đoán

Logit $[1000, 999, 998]$ ở FP32 và logit $[12, 11, 10]$ ở FP16: bản ngây thơ cho ra gì?

In [ ]:
for z, ten in ((torch.tensor([1000.0, 999.0, 998.0]), "FP32"),
               (torch.tensor([12.0, 11.0, 10.0], dtype=torch.float16), "FP16")):
    print(f"{ten}: ngây thơ {softmax_ngay_tho(z).tolist()} · ổn định "
          f"{[round(v, 4) for v in softmax_on_dinh(z).float().tolist()]}")

### Entropy chéo

Nhãn đúng là một vectơ one-hot. Với chữ số 7, số 1 nằm ở vị trí thứ tám, sau bảy số 0. Mất mát
entropy chéo $-\sum_j y_j \log \hat y_j$ khi đó chỉ còn $-\log \hat y_7$, tức chỉ phụ thuộc vào
xác suất dành cho chữ số đúng.

In [ ]:
nhan = F.one_hot(torch.tensor(7), 10).float()
assert nhan[7] == 1 and nhan.sum() == 1 and (nhan[:7] == 0).all()
print("nhãn one-hot của chữ số 7:", nhan.int().tolist())

p8 = sach(0.8, nguon="0,8 → 0,223")
p9 = sach(0.9, nguon="0,9 → 0,105")
theo_sach("−log của 0.8", -math.log(p8), sach=0.223, nguon="0,8 → 0,223")
theo_sach("−log của 0.9", -math.log(p9), sach=0.105, nguon="0,9 → 0,105")

# Lúc khởi đầu, mười lớp gần như đều nhau: xác suất của lớp đúng gần 1/10.
mat_mat_deu = F.cross_entropy(torch.zeros(1, 10), torch.tensor([7])).item()
theo_sach("mất mát khi mười lớp đều nhau, log(10)", mat_mat_deu, sach=2.3, nguon="log(10) ≈ 2,3", tol=0.05)

p = np.linspace(0.005, 1, 400)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(p, -np.log(p), color=MAU[0], lw=2)
for pi, nhan_diem, lech, canh in ((0.1, "mười lớp đều nhau", (14, 10), "left"),
                                  (p8, "ví dụ của chương", (-12, 30), "right"),
                                  (p9, "ví dụ của chương", (-12, 56), "right")):
    ax.plot([pi], [-math.log(pi)], "o", color=MAU[1], ms=8)
    ax.annotate(f"p = {pi}: {-math.log(pi):.3f} ({nhan_diem})", (pi, -math.log(pi)), xytext=lech,
                textcoords="offset points", fontsize=8, ha=canh,
                arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_ylim(0, 5.5)
ax.set_xlabel("xác suất dành cho lớp đúng")
ax.set_ylabel("mất mát entropy chéo")
ax.set_title("Xác suất của lớp đúng càng gần 0, mức phạt càng tăng vọt")
plt.tight_layout()
plt.show()

### Hai cái bẫy của công thức ngây thơ

Một softmax thiếu ổn định có thể làm tròn xác suất của lớp đúng về 0, và $\log 0 = -\infty$. Có cả
một bẫy tinh vi hơn: viết đúng dạng tổng $-\sum_j y_j \log \hat y_j$, các lớp sai có $y_j = 0$,
và nếu xác suất của chúng cũng tròn về 0 thì $0 \cdot (-\infty)$ cho ra NaN, dù lớp đúng không có
vấn đề gì.

Hiện thực ổn định của chương không bao giờ tạo ra xác suất: với $m = \max(z)$,
$\mathcal{L} = -(z_c - m) + \log\sum_j e^{z_j - m}$.

In [ ]:
def entropy_cheo_tong(z, c):
    """Đúng công thức tổng, qua softmax ngây thơ: −Σ y_j log ŷ_j."""
    return -(F.one_hot(torch.tensor(c), z.numel()).float() * torch.log(softmax_ngay_tho(z))).sum()


def entropy_cheo_on_dinh(z, c):
    m = z.max()
    return -(z[c] - m) + torch.log(torch.exp(z - m).sum())


for z, c, ghi_chu in ((torch.tensor([0.0, 200.0]), 0, "lớp đúng có xác suất tròn về 0"),
                      (torch.tensor([200.0, 0.0]), 0, "lớp sai có xác suất tròn về 0")):
    print(f"{ghi_chu}: công thức tổng {entropy_cheo_tong(z, c).item()} · ổn định "
          f"{entropy_cheo_on_dinh(z, c).item():.4g} · PyTorch {F.cross_entropy(z[None], torch.tensor([c])).item():.4g}")

dat_hat_giong()
z = torch.randn(5) * 10
assert torch.allclose(entropy_cheo_on_dinh(z, 3), F.cross_entropy(z[None], torch.tensor([3])))

### Gradient $p - y$

Khi ghép softmax với entropy chéo, gradient của mất mát theo các logit gọn tới mức đáng ngờ: xác
suất dự đoán trừ đi nhãn one-hot. Để autograd kiểm, trên một batch 32 ảnh giả lập (mất mát lấy
trung bình trên batch, nên gradient chia thêm cho 32):

In [ ]:
dat_hat_giong()
B = 32
logit = torch.randn(B, 10, requires_grad=True)
nhan_batch = torch.randint(0, 10, (B,))
F.cross_entropy(logit, nhan_batch).backward()

p_tru_y = (torch.softmax(logit.detach(), 1) - F.one_hot(nhan_batch, 10).float()) / B
lech = (logit.grad - p_tru_y).abs().max().item()
print(f"lệch lớn nhất giữa autograd và (p − y) / B: {lech:.1e}")
assert lech < 1e-7

Gradient đó lớn khi dự đoán sai mà tự tin (xác suất của lớp đúng gần 0 thì thành phần của nó gần
−1), và nhỏ dần khi dự đoán đúng. Đó là tín hiệu lỗi ban đầu mà lan truyền ngược đưa về các tầng.

## Thử thêm

1. Ở phần 1, đổi `gia_tri` thành 30,000 rồi chạy lại ô đó. Phép đổi còn hỏng không? Giá trị nào
   là giá trị nhỏ nhất làm nó hỏng?
2. Ở phần 3, thay `torch.nn.Sigmoid` bằng `torch.nn.Tanh`, hoặc tăng `so_tang` lên 20. Tầng đầu
   nhận gradient cỡ nào so với tầng cuối?
3. Ở phần 6, chạy `softmax_ngay_tho` với logit `[88.0, 0.0]` rồi `[89.0, 0.0]` ở FP32. Đối chiếu
   với ngưỡng ln của số lớn nhất đã in ở phần 2.

## Tóm lại

* Một phép tính có thể đúng cú pháp mà vẫn sai với miền giá trị của định dạng số nó dùng; thư
  viện có khi báo lỗi, có khi âm thầm trả về một số khác.
* FP16 có miền hẹp hơn FP32 nhiều: gradient nhỏ tràn dưới về 0, $e^x$ tràn trên thành Inf ngay
  khi $x$ hơn 11, và một NaN trong một mẫu làm hỏng mọi trọng số sau một bước cập nhật.
* Sigmoid và tanh bão hoà ở hai đầu, nên gradient co lại qua nhiều tầng; ReLU có đạo hàm bằng 1
  trên nhánh dương, và rẻ hơn khi hiện thực, dù tỉ lệ thời gian đo trên CPU không phải thuế
  transistor.
* Thiếu tính phi tuyến, nhiều tầng tuyến tính sụp thành một; có tầng ẩn thì biểu diễn được XOR,
  nhưng huấn luyện chưa chắc tìm ra lời giải.
* Softmax và entropy chéo ổn định về số khi trừ logit lớn nhất và không bao giờ lấy log của một
  xác suất đã tròn về 0; gradient theo logit là $p - y$.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Tính toán nơ-ron](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch05-nn-computation/), dựng từ chương
[*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_05_nn_compute.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.